In [2]:
from Model import *
from Dataset import *
from Perfomance import *

2026-04-12 20:46:14.522931: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


There is all testing for modules provided

In [11]:
#Testing functions of Model class
model = CNNLSTM()

model.compile(optimizer='SGD', loss={
        'binary': binary_loss_fn,
        'multiclass': multiclass_loss_fn
    }, metrics = [{'binary': [BinaryAUCMetric(), BinaryAccuracyMetric()], 'multiclass':AccuracyMetric()}, LossMetric()])

model([np.random.rand(5,8,8,112), np.random.rand(5)])
#model.summary()
ar = np.load('BinaryClassifierData/test.npz')
positions = ar['x']; evals = ar['evals'].astype(np.float32); target = ar['y']
#print(positions.shape, evals.shape, target.shape)
model.binary_training_step((positions, evals), target)

In [14]:
print(model.metrics[0].update_state(5))

None


In [ ]:
#Testing Dataset
ds = build_binary_dataset(100).batch(5)
positions, evals, targets = iter(ds.take(1)).next()
#print(positions.shape, evals.shape, targets.shape) None, 5, 8, 8 ,112
model.binary_call((positions, evals))

In [ ]:
#Testing Perfomance
batch_size=10

positions = np.random.randn(batch_size,4,8,8,112)
evals = np.random.randn(batch_size,4)
target = np.zeros(shape=(batch_size,num_classes))
for i in range(batch_size):
    if np.random.rand()<=class_weight:
        class_value = num_classes-1
    else:
        class_value = np.random.randint(low=0, high=num_classes-1)
    target[i, class_value]=1

'''ar = np.load("data/batch0.npz")
positions = ar['x'][:batch_size]
target = ar['y'][:batch_size]
    
evals = ar['evals'][:batch_size]
print(positions.shape, target.shape, evals.shape)
target = np.append(target, np.zeros((target.shape[0], 1)), axis=1) #For dimensionality match'''
bin_target = tf.cast(tf.equal(target[:, -1], 0), tf.int8)
with tf.GradientTape() as tape:
    preds = model((positions, evals))
    binary_loss = binary_loss_fn(bin_target, preds['binary'])
    multiclass_loss = multiclass_loss_fn(target, preds['multiclass'])
    loss = binary_loss+multiclass_loss

grad = tape.gradient(loss, model.trainable_variables)
#print(f"\n\n Preds are {preds} \n\n")

binary_auc = BinaryAUCMetric()
binary_auc.update_state(target, preds)
print(f"Binary AUC is {binary_auc.result()}")
mean_loss = LossMetric()
mean_loss.update_state(loss)
print(f"Mean loss is unexpectedly {mean_loss.result}")
#model.evaluate()
binary_acc = BinaryAccuracyMetric()
binary_acc.update_state(target, preds)
print(f"Binary accuracy is {binary_acc.result()}")
multiclass_acc = AccuracyMetric()
multiclass_acc.update_state(target, preds)
print(f"Class prediction accuracy is {multiclass_acc.result()}")

#model.evaluate(x=(positions, evals), y={'binary':target, 'multiclass':target},verbose=1)